# SCAMPER vs PULSE vs MIMIC-MJX: Gait Metrics Comparison (S3e-1 Walk)

Side-by-side comparison of gait metrics from SCAMPER, PULSE, and MIMIC-MJX rollouts against real rat lateral-sequence walk data.

- **Δσ** = |mean_model − mean_rat| / √(σ²_model + σ²_rat)
- Lower Δσ means the model's gait is closer to the real rat.

In [6]:
import pandas as pd
import numpy as np

# ── Data extracted from the three analysis notebooks ────────────────────────
# Columns: Metric, Table 1 #, model mean±std string, rat mean±std string, Δσ
# SCAMPER   notebook: maintain_velocity_analysis_S3e-1_walk_SCAMPER.ipynb  (cell 40)
# PULSE     notebook: maintain_velocity_analysis_S3e-1_walk_PULSE.ipynb    (cell 40)
# MIMIC-MJX notebook: maintain_velocity_analysis_S3e-1_walk_MIMICMJX.ipynb (cell 40)

metrics = [
    "Stride Duration (ms)",
    "Step Frequency (Hz)",
    "Stance Duration LH (ms)",
    "Stance Duration RH (ms)",
    "Stance Duration LF (ms)",
    "Stance Duration RF (ms)",
    "Swing Duration LH (ms)",
    "Swing Duration RH (ms)",
    "Swing Duration LF (ms)",
    "Swing Duration RF (ms)",
    "Duty Factor LH",
    "Duty Factor RH",
    "Duty Factor LF",
    "Duty Factor RF",
    "Temporal Symmetry (hind)",
    "Hindlimb Phase Lag (RH rel. LH)",
    "Homolateral Coupling (LF rel. LH)",
    "Diagonal Coupling (RF rel. LH)",
]

table1_nums = [4, 15, 17, 17, 17, 17, 18, 18, 18, 18, 6, 6, 6, 6, 7, 16, 20, 19]

# SCAMPER rollout values
scamper_str = [
    "392 ± 27", "2.57 ± 0.38",
    "220 ± 46", "219 ± 40", "186 ± 23", "240 ± 9",
    "166 ± 35", "169 ± 20", "203 ± 5", "155 ± 7",
    "0.562 ± 0.118", "0.559 ± 0.102", "0.474 ± 0.059", "0.613 ± 0.022",
    "-0.0092 ± 0.0773",
    "0.455 ± 0.094", "0.393 ± 0.080", "0.823 ± 0.107",
]
scamper_sigma = [0.90, 1.04, 0.41, 0.58, 0.20, 0.96, 1.01, 0.99, 2.64, 0.69,
                 0.10, 0.03, 0.84, 0.92, 0.20, 0.04, 0.65, 0.01]

# PULSE rollout values
pulse_str = [
    "332 ± 16", "3.02 ± 0.14",
    "191 ± 24", "184 ± 9", "173 ± 34", "183 ± 5",
    "144 ± 6", "147 ± 6", "151 ± 6", "149 ± 4",
    "0.576 ± 0.071", "0.555 ± 0.027", "0.521 ± 0.101", "0.552 ± 0.014",
    "0.0122 ± 0.0282",
    "0.510 ± 0.116", "0.451 ± 0.098", "0.900 ± 0.158",
]
pulse_sigma = [0.27, 0.57, 0.11, 0.14, 0.03, 0.13, 0.92, 0.46, 0.56, 0.50,
               0.00, 0.12, 0.20, 0.11, 0.02, 0.41, 1.10, 0.46]

# MIMIC-MJX rollout values
mimicmjx_str = [
    "378 ± 24", "2.66 ± 0.25",
    "233 ± 9", "247 ± 28", "142 ± 18", "198 ± 7",
    "151 ± 10", "125 ± 5", "234 ± 22", "182 ± 6",
    "0.616 ± 0.023", "0.652 ± 0.075", "0.375 ± 0.047", "0.522 ± 0.020",
    "-0.0430 ± 0.0301",
    "0.369 ± 0.095", "0.532 ± 0.100", "0.816 ± 0.213",
]
mimicmjx_sigma = [0.75, 1.00, 0.62, 1.00, 0.40, 0.34, 1.15, 0.23, 2.94, 1.56,
                  0.52, 0.96, 2.26, 0.61, 0.72, 0.88, 1.80, 0.03]

# Real rat values (same in all notebooks)
rat_str = [
    "308 ± 91", "3.46 ± 0.75",
    "182 ± 82", "175 ± 66", "171 ± 71", "174 ± 69",
    "125 ± 20", "133 ± 31", "136 ± 25", "133 ± 30",
    "0.576 ± 0.074", "0.562 ± 0.055", "0.544 ± 0.059", "0.558 ± 0.055",
    "0.0110 ± 0.0682",
    "0.459 ± 0.041", "0.333 ± 0.047", "0.824 ± 0.045",
]

## Comparison Table: Model vs Real Rat (Δσ)

Each model's Δσ measures how far its gait metric is from the real rat in units of combined standard deviation. The **Winner** column shows which model is closest to the rat (lowest Δσ), highlighted in the styled table below.

In [7]:
# Build the three-way comparison DataFrame
winner = []
for s, p, m in zip(scamper_sigma, pulse_sigma, mimicmjx_sigma):
    best = min(s, p, m)
    names = []
    if abs(s - best) < 0.005:
        names.append("SCAMPER")
    if abs(p - best) < 0.005:
        names.append("PULSE")
    if abs(m - best) < 0.005:
        names.append("MIMIC-MJX")
    winner.append(" / ".join(names) if names else "Tie")

df = pd.DataFrame({
    "Metric": metrics,
    "Table 1 #": table1_nums,
    "SCAMPER (mean ± std)": scamper_str,
    "Δσ SCAMPER": scamper_sigma,
    "PULSE (mean ± std)": pulse_str,
    "Δσ PULSE": pulse_sigma,
    "MIMIC-MJX (mean ± std)": mimicmjx_str,
    "Δσ MIMIC-MJX": mimicmjx_sigma,
    "Real Rat (mean ± std)": rat_str,
    "Closest to Rat": winner,
})

df

,Metric,Table 1 #,SCAMPER (mean ± std),Δσ SCAMPER,PULSE (mean ± std),Δσ PULSE,MIMIC-MJX (mean ± std),Δσ MIMIC-MJX,Real Rat (mean ± std),Closest to Rat
0,Stride Duration (ms),4,392 ± 27,0.90,332 ± 16,0.27,378 ± 24,0.75,308 ± 91,PULSE
1,Step Frequency (Hz),15,2.57 ± 0.38,1.04,3.02 ± 0.14,0.57,2.66 ± 0.25,1.00,3.46 ± 0.75,PULSE
2,Stance Duration LH (ms),17,220 ± 46,0.41,191 ± 24,0.11,233 ± 9,0.62,182 ± 82,PULSE
3,Stance Duration RH (ms),17,219 ± 40,0.58,184 ± 9,0.14,247 ± 28,1.00,175 ± 66,PULSE
4,Stance Duration LF (ms),17,186 ± 23,0.20,173 ± 34,0.03,142 ± 18,0.40,171 ± 71,PULSE
5,Stance Duration RF (ms),17,240 ± 9,0.96,183 ± 5,0.13,198 ± 7,0.34,174 ± 69,PULSE
6,Swing Duration LH (ms),18,166 ± 35,1.01,144 ± 6,0.92,151 ± 10,1.15,125 ± 20,PULSE
7,Swing Duration RH (ms),18,169 ± 20,0.99,147 ± 6,0.46,125 ± 5,0.23,133 ± 31,MIMIC-MJX
8,Swing Duration LF (ms),18,203 ± 5,2.64,151 ± 6,0.56,234 ± 22,2.94,136 ± 25,PULSE
9,Swing Duration RF (ms),18,155 ± 7,0.69,149 ± 4,0.50,182 ± 6,1.56,133 ± 30,PULSE


In [8]:
# Styled version: highlight the lowest Δσ in green per row
def highlight_lowest_sigma(row):
    """Apply green background to the lowest Δσ column(s) in each row."""
    styles = [""] * len(row)
    s_col = row.index.get_loc("Δσ SCAMPER")
    p_col = row.index.get_loc("Δσ PULSE")
    m_col = row.index.get_loc("Δσ MIMIC-MJX")
    s_val, p_val, m_val = row["Δσ SCAMPER"], row["Δσ PULSE"], row["Δσ MIMIC-MJX"]
    green = "background-color: #c6efce; font-weight: bold"
    best = min(s_val, p_val, m_val)
    if abs(s_val - best) < 0.005:
        styles[s_col] = green
    if abs(p_val - best) < 0.005:
        styles[p_col] = green
    if abs(m_val - best) < 0.005:
        styles[m_col] = green
    return styles

df.style.apply(highlight_lowest_sigma, axis=1).format(
    {"Δσ SCAMPER": "{:.2f}", "Δσ PULSE": "{:.2f}", "Δσ MIMIC-MJX": "{:.2f}"}
).set_caption("SCAMPER vs PULSE vs MIMIC-MJX: Δσ distance to real rat gait metrics")

,Metric,Table 1 #,SCAMPER (mean ± std),Δσ SCAMPER,PULSE (mean ± std),Δσ PULSE,MIMIC-MJX (mean ± std),Δσ MIMIC-MJX,Real Rat (mean ± std),Closest to Rat
0,Stride Duration (ms),4,392 ± 27,0.90,332 ± 16,0.27,378 ± 24,0.75,308 ± 91,PULSE
1,Step Frequency (Hz),15,2.57 ± 0.38,1.04,3.02 ± 0.14,0.57,2.66 ± 0.25,1.00,3.46 ± 0.75,PULSE
2,Stance Duration LH (ms),17,220 ± 46,0.41,191 ± 24,0.11,233 ± 9,0.62,182 ± 82,PULSE
3,Stance Duration RH (ms),17,219 ± 40,0.58,184 ± 9,0.14,247 ± 28,1.00,175 ± 66,PULSE
4,Stance Duration LF (ms),17,186 ± 23,0.20,173 ± 34,0.03,142 ± 18,0.40,171 ± 71,PULSE
5,Stance Duration RF (ms),17,240 ± 9,0.96,183 ± 5,0.13,198 ± 7,0.34,174 ± 69,PULSE
6,Swing Duration LH (ms),18,166 ± 35,1.01,144 ± 6,0.92,151 ± 10,1.15,125 ± 20,PULSE
7,Swing Duration RH (ms),18,169 ± 20,0.99,147 ± 6,0.46,125 ± 5,0.23,133 ± 31,MIMIC-MJX
8,Swing Duration LF (ms),18,203 ± 5,2.64,151 ± 6,0.56,234 ± 22,2.94,136 ± 25,PULSE
9,Swing Duration RF (ms),18,155 ± 7,0.69,149 ± 4,0.50,182 ± 6,1.56,133 ± 30,PULSE


## Summary

Win counts and average Δσ for each method.

In [9]:
n_scamper = sum(1 for w in winner if "SCAMPER" in w)
n_pulse = sum(1 for w in winner if "PULSE" in w)
n_mimicmjx = sum(1 for w in winner if "MIMIC-MJX" in w)

print(f"Metrics where SCAMPER   is closest to rat: {n_scamper}/{len(metrics)}")
print(f"Metrics where PULSE     is closest to rat: {n_pulse}/{len(metrics)}")
print(f"Metrics where MIMIC-MJX is closest to rat: {n_mimicmjx}/{len(metrics)}")
print()
print(f"Mean Δσ  SCAMPER:   {np.mean(scamper_sigma):.3f}")
print(f"Mean Δσ  PULSE:     {np.mean(pulse_sigma):.3f}")
print(f"Mean Δσ  MIMIC-MJX: {np.mean(mimicmjx_sigma):.3f}")

Metrics where SCAMPER   is closest to rat: 4/18
Metrics where PULSE     is closest to rat: 13/18
Metrics where MIMIC-MJX is closest to rat: 1/18

Mean Δσ  SCAMPER:   0.678
Mean Δσ  PULSE:     0.339
Mean Δσ  MIMIC-MJX: 0.987
